In [1]:
from import_dados import *
import polars_ds as pds
pl.Config.set_tbl_rows(-1)

In [2]:
sinistros_bruto = read_sinistros()

# Considerações iniciais

Para a análise estamos considerando apenas os registros ocorridos em vias públicas e que tenham um tipo de sinistro definido segundo as regras no dicionário de dados disponibilizado.
Sendo assim, os registros com tp_sinistro_primario = 'NAO DISPONIVEL' não serão considerados

O agrupamento do tipo de sinistro primário é
definido seguindo a priorização (de 1 a 14)
dos seus respectivos tipos de sinistro:
- ATROPELAMENTO: 1 - pedestre, 2 - vítima
fora do veículo;
- COLISAO: 3 - frontal, 4 - traseira,
5 - lateral, 6 - transversal, 7 - outros;
- CHOQUE: 8;
- OUTROS: 9 - capotamento, 10 -
engavetamento, 11 - tombamento, 12 -
atropelamento animal, 13 - outros;

In [3]:
sinistros = sinistros_bruto.filter(pl.col('tipo_local') == 'PUBLICO') \
                     .with_columns(pl.col('hora_sinistro').str.split_exact(by=':', n=1)
                                   .struct.rename_fields(['hora', 'minuto']).alias('hora_struct')
                                     ).unnest('hora_struct') \
                     .with_columns(pl.col('hora').cast(pl.Int16), 
                                   pl.col('minuto').cast(pl.Int16),
                                   pl.when(pl.col('tipo_registro') == 'SINISTRO FATAL').then(pl.lit('SIM')).otherwise(pl.lit('NAO')).alias('acidente_fatal'))

# Primeira análise

Analisando os dados dos sinistros, encontramos registros com o tipo de via 'NAO DISPONIVEL'   
Todos os registros dessa categoria são sinalizados como fatais, sendo um dado que não reflete a realidade e distorce a associação entre tipo de via e fatalidade. Denotando um viés de seleção para a categoria  
A representatividade dos registros é muito baixa, com apenas 0,08% do total, porém tem um impacto grande na análise de Qui quadrado.  
A remoção dos registros "NAO DISPONIVEL" reduz o Qui quadrado em aproximadamente 33.9% (de 8017 para 5299)
Portanto a retirada destes registros deve ser realizada dos registros.

In [5]:
result_total = sinistros \
.select(pds.chi2("tipo_via", "acidente_fatal").alias("teste_chi2")) \
.unnest("teste_chi2")

result_filtrado = sinistros \
.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
.select(pds.chi2("tipo_via", "acidente_fatal").alias("teste_chi2")) \
.unnest("teste_chi2")

print(result_total)
print(result_filtrado)

sinistros = sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL')

In [ ]:
chave_fill = []
media_por_grupo = sinistros.group_by([
    'dia_da_semana', 'tipo_registro', 'tipo_via', 
    'regiao_administrativa', 'tp_sinistro_primario'
]).agg(
    pl.mean('hora').alias('hora_media')
)

sinistros.join(media_por_grupo, ).with_columns(
    pl.coalesce(pl.col('hora'), pl.mean('hora').over(partition_by=['dia_da_semana', 'tipo_registro', 'tipo_via', 'regiao_administrativa', 'tp_sinistro_primario']).floor().cast(pl.Int16)).alias('hora'))

In [6]:
sinistros.group_by('dia_da_semana', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .plot.bar(x=alt.X(**get_col_order('dia_da_semana')), 
              y=alt.Y('acidentes').stack(None), 
              row=alt.Row(**get_col_order('acidente_fatal')), 
              column='tipo_via') \
    .properties(height=150, width=350) \
    #.resolve_scale(y='independent')
    

In [98]:
sinistros.filter(pl.col('tipo_via') != ND) \
    .filter(pl.col('turno') != ND) \
    .group_by('hora', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.bar(x='hora', 
              y=alt.Y('acidentes').stack(None), 
              column=alt.Column(**get_col_order('acidente_fatal')), 
              row='tipo_via') \
    .properties(height=150, width=350) \
    .resolve_scale(y='independent')

In [99]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('hora').is_not_null()) \
    .group_by('dia_da_semana', 'hora', 'tipo_via', 'acidente_fatal').agg(pl.len().alias('acidentes')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y=alt.Y(**get_col_order('dia_da_semana')), row='tipo_via', column=alt.Column(**get_col_order('acidente_fatal'))) \
    .properties(height=150, width=500) \
    .resolve_scale(color='independent')


In [100]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('hora').is_not_null()) \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'hora', 'tipo_via').agg(pl.len().alias('acidentes')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y=alt.Y(**get_col_order('dia_da_semana')), row='tipo_via') \
    .properties(height=150, width=500)

In [101]:
sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('hora').is_not_null()) \
    .with_columns((pl.when(pl.col('acidente_fatal') == 'SIM').then(1).otherwise(0)).alias('acidente_fatal')) \
    .group_by('dia_da_semana', 'hora', 'tipo_via') \
    .agg(
        (pl.sum('acidente_fatal') *100 / pl.count('acidente_fatal')).alias('taxa_de_fatalidade')
    ) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='taxa_de_fatalidade', y=alt.Y(**get_col_order('dia_da_semana')), row='tipo_via').properties(height=150, width=550) \
        .resolve_scale(color='independent')


In [102]:

sinistros.filter(pl.col('tipo_via') != 'NAO DISPONIVEL') \
    .filter(pl.col('hora').is_not_null()) \
    .filter(pl.col('acidente_fatal') == 'SIM') \
    .group_by('dia_da_semana', 'hora', 'tipo_via', 'tp_sinistro_primario').agg(pl.len().alias('acidentes')) \
    .with_columns(pl.col('hora').cast(pl.String).str.zfill(2)) \
    .plot.rect(x='hora', color='acidentes', y=alt.Y(**get_col_order('dia_da_semana')), column='tipo_via', row='tp_sinistro_primario') \
    .properties(height=150, width=500)